# Build Vector Index on Colab Pro GPU

Run this notebook on Colab Pro (T4 or better) to:
1. Clone your repo
2. Download FDA Orange Book
3. Build the ChromaDB index with bge-m3 on GPU (~30 sec vs ~25 min on CPU)
4. Save the ChromaDB folder to Google Drive
5. Download it to your local machine for fast querying

**Prerequisites:**
- Runtime → Change runtime type → T4 GPU
- Google Drive mounted

## 1. Mount Google Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this path to where you want ChromaDB saved on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/pharma-intel/chroma_db'

# If your repo is on GitHub:
# !git clone https://github.com/YOUR_USER/pharma-intelligence.git
# %cd pharma-intelligence

# Or upload the zip manually via Colab's file browser and unzip:
# !unzip pharma-intelligence.zip && cd pharma-intelligence

## 2. Install dependencies

In [ ]:
!pip install -q -e .

## 3. Set env vars (ThaiLLM API key)

⚠️ Do NOT commit the key. Use Colab's "Secrets" feature (🔑 icon in left sidebar) to store `THAILLM_API_KEY`, then load here:

In [ ]:
import os
from google.colab import userdata

os.environ['THAILLM_API_KEY'] = userdata.get('THAILLM_API_KEY')
os.environ['LLM_PROVIDER'] = 'thaillm'

# Save ChromaDB directly on Drive
os.environ['CHROMA_PERSIST_DIR'] = DRIVE_OUTPUT
os.environ['DATA_DIR'] = '/content/drive/MyDrive/pharma-intel/data'
os.environ['MODEL_CACHE_DIR'] = '/content/drive/MyDrive/pharma-intel/.cache/models'

## 4. Verify GPU is available

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

## 5. Run the pipeline

In [ ]:
from pharma_intel.ingestion.fda_orange_book import run_pipeline
from pharma_intel.rag.indexer import index_monographs

# Step 1: download + parse + filter
monographs = run_pipeline(force_download=False)
print(f'\n✓ {len(monographs)} monographs ready for indexing')

In [ ]:
# Step 2: embed with bge-m3 on GPU — this is the speedup
import time
start = time.time()
store = index_monographs(monographs, batch_size=64)   # larger batch on GPU
print(f'\n✓ Indexing took {time.time() - start:.1f} seconds')
print(f'✓ Collection size: {store.collection.count()}')

## 6. Test query on Colab

In [ ]:
from pharma_intel.rag import RAGEngine

engine = RAGEngine(top_k=5)
result = engine.answer('What patents cover empagliflozin and when do they expire?')

print('--- ANSWER ---')
print(result.answer)
print(f'\n--- Retrieved {len(result.retrieved)} sources ---')
for c in result.retrieved:
    print(f'  [{c.doc_id}] {c.metadata["ingredient"]} — score={c.score:.3f}')

## 7. ChromaDB is already on Drive — pull to local

Since we set `CHROMA_PERSIST_DIR` to the Drive path, the index is already there.

On your local machine, mount Drive (e.g., with Google Drive Desktop) or download
the `chroma_db/` folder, then run locally:

```bash
export CHROMA_PERSIST_DIR=/path/to/downloaded/chroma_db
python scripts/query_rag.py 'your question'
```

Local querying is fast (<1 sec per query) because only 1 embedding is computed per query.